In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import sha2, concat_ws, col, when

## @params: [JOB_NAME]
# args = getResolvedOptions(sys.argv, ['JOB_NAME'])

sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

dim_product_df = spark.read.csv("s3://retail-data-142083400213/dimension/products.csv", header=True)
dim_product_df = dim_product_df.withColumn("product_id", sha2(concat_ws("||", "product_name", "brand"),256))
dim_product_df.show()

dim_customer_df = spark.read.csv("s3://retail-data-142083400213/dimension/customer.csv", header=True)
dim_customer_df = dim_customer_df.withColumn("customer_id", sha2(concat_ws("||", "name", "city", "state"),256))
dim_customer_df.show()

dim_store_df = spark.read.csv("s3://retail-data-142083400213/dimension/stores.csv", header=True)
dim_store_df = dim_store_df.withColumn("store_id", sha2("store_name",256))
dim_store_df.show()

sales_df = spark.read.csv("s3://retail-data-142083400213/sales/sales.csv", header=True)


s = sales_df.alias("s")
p = dim_product_df.alias("p")
st = dim_store_df.alias("st")
c = dim_customer_df.alias("c")

sales_df = (
    s
    .join(p,(col("s.product") == col("p.product_name")) & (col("s.brand") == col("p.brand")), how='left')
    .join(c,(col("s.name") == col("c.name")) & (col("s.city") == col("c.city")) & (col("s.region") == col("c.state")), how='left')
    .join(st,(col("s.store") == col("st.store_name")),how='left')
    .select(
        col("s.invoice_no"),
        col("c.customer_id"),
        col("p.product_id"),
        col("p.cost_price"),
        col("st.store_id"),
        col("s.quantity"),
        col("s.unit_price"),
        col("s.discount"),
        col("s.order_timestamp"),
        col("s.payment_mode")
    )
)

# CAST --> Used to convert one data type to another data type
# str, int, double, float
sales_df = sales_df \
            .withColumn("quantity", col("quantity").cast("int")) \
            .withColumn("unit_price", col("unit_price").cast("int")) \
            .withColumn("discount", col("discount").cast("int")) \
            .withColumn("cost_price", col("cost_price").cast("int"))
            
#Calculation for Gross amount, net amount, profit and profit_percentage
sales_df = sales_df \
            .withColumn("gross_amount", col("quantity") * col("unit_price")) \
            .withColumn("net_amount", col("gross_amount") - col("discount")) \
            .withColumn("profit",col("net_amount") - (col("cost_price") * col("quantity"))) \
            .withColumn("profit_percentage", 
                when(col("profit") != 0, 
                (col("profit") / col("net_amount")) * 100).otherwise(0))
                
# Derive the segment column (High Value, Medium Value and low value)
sales_df = sales_df.withColumn(
                        "segment",
                        when(col("net_amount") > 50000, "High Value")
                        .when(col("net_amount") > 10000, "Medium Value")
                        .otherwise("Low Value")
                    )
# sales_df.show()
# 
high_value_sales = sales_df.filter(col("segment") == "High Value")
# high_value_sales.show()

# Get the high margin products
high_margin_sales = sales_df.filter(col("profit_percentage") > 30)
high_margin_products = (
    high_margin_sales
    .join(dim_product_df, on="product_id", how="left")
    .select("product_name", "brand")
)
high_margin_products.show()

# Write to S3 bucket in parqute
sales_df.write \
    .mode("overwrite") \
    .parquet("s3://retail-data-142083400213/output/parquet")
    

# job = Job(glueContext)
# job.init(args['JOB_NAME'], args)
# job.commit()